# Renewable Asset Performance Dashboard with Anomaly Detection

**Domain:** Energy / Data Engineering / Analytics  
**Primary Evaluation Focus:** Capacity Factor, Generation, Anomaly Rate  
**Dataset:** Synthetic wind and solar SCADA-style asset data

##  Executive Summary

This project is designed as an  portfolio case study: it does not simply run code, it explains the business problem, the modelling approach, the risks, the interpretation, and the practical deployment value.

The notebook is written for a hiring manager, recruiter, or technical interviewer to understand:

- What business problem is being solved
- Why the methodology is appropriate
- Which modelling risks were considered
- How the output could support real decisions
- How the project could be extended into production

## Key Findings

- Asset-level monitoring can identify underperformance across wind and solar portfolios.
- Rolling z-score anomaly detection highlights operational issues early.
- Capacity factor provides a commercially interpretable performance metric.
- Regional and asset-type segmentation improves root-cause analysis.
- Executive dashboards convert technical SCADA data into actionable portfolio insight.

## Business Recommendations

- Prioritise maintenance for assets with persistent negative anomalies.
- Track capacity factor trends by region and technology.
- Create automated alerts for underperforming assets.
- Integrate weather-normalised baselines in a production version.
- Use dashboard outputs for operations and asset-management reviews.

## 

This  places emphasis on:

- Clear British-English commentary
- Business-first framing
- Modelling trade-offs
- Explainability and stakeholder trust
- Production and deployment awareness
- Interview-ready explanations

# Methodology and Modelling Rationale

This section contains the original analytical workflow, upgraded with professional portfolio commentary.

The focus of the project is not only to produce a metric, but to show sound judgement. In a commercial data role, the strongest candidates are able to explain:

- Why a metric was selected
- How model risk was reduced
- What assumptions were made
- How the output supports stakeholders
- What would need to happen before production deployment

For this project, the primary evaluation focus is: **Capacity Factor, Generation, Anomaly Rate**.

# Project 06 - Renewable Asset Performance Dashboard
**Domain:** Energy / Data Engineering

End-to-end pipeline monitoring 240 wind and solar assets with ETL, anomaly detection, and executive dashboard.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
np.random.seed(42)
print("Ready")

In [ ]:
# ── SIMULATE SCADA DATA ────────────────────────────────────────
n_assets = 240
n_days   = 365

records = []
for asset_id in range(1, n_assets + 1):
    asset_type = 'Wind' if asset_id <= 120 else 'Solar'
    region     = np.random.choice(['Scotland','Wales','England North','England South','Offshore'])
    cap_mw     = np.random.uniform(2, 8)
    dates      = pd.date_range('2023-01-01', periods=n_days, freq='D')

    if asset_type == 'Solar':
        month_factor = np.array([0.4,0.5,0.7,0.9,1.0,1.0,1.0,0.95,0.8,0.6,0.4,0.35])
        gen = np.array([cap_mw * 4.5 * month_factor[d.month-1] * np.random.uniform(0.5,1.0)
                        for d in dates])
    else:
        gen = np.array([cap_mw * np.random.uniform(3, 18) for _ in dates])

    downtime_idx = np.random.choice(n_days, size=int(n_days*0.008), replace=False)
    gen[downtime_idx] = 0.0
    anomaly_idx = np.random.choice(n_days, size=int(n_days*0.002), replace=False)
    gen[anomaly_idx] *= 0.1

    for i, d in enumerate(dates):
        records.append({'date': d, 'asset_id': asset_id, 'asset_type': asset_type,
                        'region': region, 'capacity_mw': round(cap_mw,2),
                        'generation_mwh': round(gen[i], 3)})

df = pd.DataFrame(records)
df['capacity_factor'] = df['generation_mwh'] / (df['capacity_mw'] * 24)
print(f"SCADA records: {len(df):,}")
print(f"Total generation: {df['generation_mwh'].sum()/1000:,.0f} GWh")
df.head()

In [ ]:
# ── SQL-STYLE AGGREGATIONS ─────────────────────────────────────
asset_summary = df.groupby(['asset_id','asset_type','region','capacity_mw']).agg(
    total_mwh=('generation_mwh','sum'),
    avg_cap_factor=('capacity_factor','mean'),
    zero_days=('generation_mwh', lambda x: (x<0.1).sum())
).reset_index()
asset_summary['uptime_pct'] = (1 - asset_summary['zero_days']/365) * 100

print("Fleet Performance Summary:")
print(asset_summary.groupby('asset_type').agg({'avg_cap_factor':'mean','uptime_pct':'mean','total_mwh':'sum'}).round(3))

In [ ]:
# ── ANOMALY DETECTION (Z-SCORE) ────────────────────────────────
df = df.sort_values(['asset_id','date'])
df['roll_mean'] = df.groupby('asset_id')['generation_mwh'].transform(lambda x: x.rolling(30, min_periods=5).mean())
df['roll_std']  = df.groupby('asset_id')['generation_mwh'].transform(lambda x: x.rolling(30, min_periods=5).std())
df['z_score']   = (df['generation_mwh'] - df['roll_mean']) / (df['roll_std'] + 0.001)
df['anomaly']   = (df['z_score'] < -3).astype(int)

print(f"Anomalies detected: {df['anomaly'].sum():,} ({df['anomaly'].mean():.2%})")
print(f"Affected assets: {df[df['anomaly']==1]['asset_id'].nunique()}")

In [ ]:
# ── EXECUTIVE DASHBOARD ────────────────────────────────────────
fig = plt.figure(figsize=(18, 12))
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.3)

kpi_ax = fig.add_subplot(gs[0,:])
kpi_ax.axis('off')
kpis = [
    ("Total Generation", f"{df['generation_mwh'].sum()/1000:,.0f} GWh"),
    ("Fleet Avg Cap Factor", f"{df['capacity_factor'].mean():.1%}"),
    ("Fleet Uptime", f"{asset_summary['uptime_pct'].mean():.1f}%"),
    ("Anomalies", f"{df['anomaly'].sum():,}"),
    ("Assets", "240"),
]
for i,(label,val) in enumerate(kpis):
    x = 0.04 + i*0.19
    kpi_ax.add_patch(plt.Rectangle((x,0.05),0.17,0.9,color='#0F1C2E',transform=kpi_ax.transAxes,zorder=1))
    kpi_ax.text(x+0.085,0.65,val,ha='center',va='center',fontsize=13,fontweight='bold',color='#C9A96E',transform=kpi_ax.transAxes,zorder=2)
    kpi_ax.text(x+0.085,0.25,label,ha='center',va='center',fontsize=8,color='white',transform=kpi_ax.transAxes,zorder=2)

ax1 = fig.add_subplot(gs[1,:2])
monthly = df.groupby([df['date'].dt.month,'asset_type'])['generation_mwh'].sum().unstack().fillna(0)
monthly.plot(kind='bar', ax=ax1, color=['#C9A96E','#0F1C2E'], edgecolor='white')
ax1.set_title('Monthly Generation by Asset Type (MWh)', fontweight='bold')
ax1.set_xlabel('Month')

ax2 = fig.add_subplot(gs[1,2])
region_pivot = df.groupby(['region','asset_type'])['generation_mwh'].sum().unstack().fillna(0)
sns.heatmap(region_pivot/1000, annot=True, fmt='.0f', cmap='YlOrBr', ax=ax2)
ax2.set_title('Generation GWh by Region', fontweight='bold')

ax3 = fig.add_subplot(gs[2,:2])
daily_anomalies = df.groupby('date')['anomaly'].sum()
ax3.plot(daily_anomalies.index, daily_anomalies.values, color='#C9A96E', lw=1)
ax3.fill_between(daily_anomalies.index, daily_anomalies.values, alpha=0.3, color='#C9A96E')
ax3.set_title('Daily Anomaly Count', fontweight='bold')

ax4 = fig.add_subplot(gs[2,2])
for atype, grp in asset_summary.groupby('asset_type'):
    ax4.hist(grp['avg_cap_factor'], bins=20, alpha=0.7,
             color='#C9A96E' if atype=='Solar' else '#0F1C2E', label=atype)
ax4.set_title('Capacity Factor Distribution', fontweight='bold')
ax4.legend()

fig.suptitle('Renewable Asset Performance Dashboard — 240 Assets 2023', fontsize=16, fontweight='bold', y=1.01)
plt.savefig('renewable_dashboard.png', dpi=120, bbox_inches='tight')
plt.show()
print("Dashboard saved as renewable_dashboard.png")

# Final Business Interpretation

## Key Findings

- Asset-level monitoring can identify underperformance across wind and solar portfolios.
- Rolling z-score anomaly detection highlights operational issues early.
- Capacity factor provides a commercially interpretable performance metric.
- Regional and asset-type segmentation improves root-cause analysis.
- Executive dashboards convert technical SCADA data into actionable portfolio insight.

## Recommended Next Steps

- Prioritise maintenance for assets with persistent negative anomalies.
- Track capacity factor trends by region and technology.
- Create automated alerts for underperforming assets.
- Integrate weather-normalised baselines in a production version.
- Use dashboard outputs for operations and asset-management reviews.

## Interview Talking Point

A strong way to discuss this project in an interview:

> "Developed a renewable asset performance dashboard simulating 240 wind and solar assets, using ETL-style aggregation and anomaly detection to support operational decision-making."

## Production Considerations

Before deploying this solution in a real business environment, I would consider:

- Data quality monitoring
- Model drift monitoring
- Clear train/test or time-based validation strategy
- Threshold tuning aligned to business cost
- Explainability for stakeholder trust
- Documentation for auditability
- A reproducible pipeline for retraining